# M3L2 E00 - Del agente manual (M3L1) a LangChain (Resolution)


## Tabla de mapeo M3L1 → M3L2

| M3L1 manual | LangChain M3L2 |
|---|---|
| `def weather_tool(city)` | `@tool def weather_tool(city: str) -> str` |
| `choose_action(state)` hardcodeado | LLM decide con tool binding |
| `for step in range(max_steps)` | `max_iterations=5` en AgentExecutor |
| `trace = []` + prints manuales | `verbose=True` en AgentExecutor |
| `[Thought]/[Action]/[Obs]` a mano | Mensajes internos del AgentExecutor |
| `state = {}` manual | Historial del AgentExecutor |


In [ ]:
# Agente manual M3L1 (referencia, no cambiar)
WEATHER_DB = {"Paris": 21, "London": 15, "Berlin": 18, "Buenos Aires": 25}

def weather_tool_manual(city):
    temp = WEATHER_DB.get(city)
    if temp is None:
        return {"success": False, "error": f"No data for {city}"}
    return {"success": True, "city": city, "temperatura_c": temp}

def calculator_tool_manual(operacion, a, b):
    ops = {"multiplicar": lambda x,y: x*y, "sumar": lambda x,y: x+y}
    if operacion not in ops:
        return {"success": False, "error": "Operacion no soportada"}
    return {"success": True, "resultado": ops[operacion](a, b)}

def thought(m): print(f"  [Thought] {m}")
def action(t, *a): print(f"  [Action]  {t}({', '.join(str(x) for x in a)})")
def observation(r): print(f"  [Observation] {r}")
def final_answer(r): print(f"  [Final Answer] {r}"); return r

def agente_manual(consulta, ciudad):
    print(f"\n[Consulta] {consulta}\n")
    thought(f"Necesito la temperatura de {ciudad}.")
    action("weather_tool", ciudad)
    rc = weather_tool_manual(ciudad)
    observation(rc)
    if not rc["success"]:
        return final_answer(f"No pude obtener el clima de {ciudad}.")
    temp = rc["temperatura_c"]
    thought(f"Ahora debo multiplicar {temp} por 5.")
    action("calculator_tool", "multiplicar", temp, 5)
    rk = calculator_tool_manual("multiplicar", temp, 5)
    observation(rk)
    return final_answer(f"La temperatura en {ciudad} es {temp}C y cinco veces eso es {rk['resultado']}C.")

print("=" * 55)
print("AGENTE MANUAL (Estilo M3L1)")
print("=" * 55)
agente_manual("Cual es el clima en Paris y 5 veces esa temperatura?", "Paris")


In [ ]:
import os, getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [ ]:
# TODO 1: @tool decorator con docstrings claros

@tool
def weather_tool(city: str) -> str:
    """
    Get the current temperature for a given city in Celsius.
    Use this tool when the user asks about the weather or temperature of a city.
    Returns the temperature as a string like '21 Celsius'.
    """
    temps = {"Paris": 21, "London": 15, "Berlin": 18, "Buenos Aires": 25}
    temp = temps.get(city)
    if temp is None:
        return f"No temperature data available for {city}"
    return f"{temp} Celsius"


@tool
def calculator_tool(operation: str, a: float, b: float) -> float:
    """
    Perform a safe arithmetic calculation.
    Supported operations: multiply, add, subtract, divide.
    Use this tool for any mathematical calculation to ensure accuracy.
    """
    ops = {"multiply": lambda x,y: x*y, "add": lambda x,y: x+y, "subtract": lambda x,y: x-y}
    if operation == "divide":
        if b == 0: raise ValueError("Cannot divide by zero")
        return a / b
    if operation not in ops:
        raise ValueError(f"Unsupported operation: {operation}. Use: multiply, add, subtract, divide")
    return ops[operation](a, b)


print(f"weather_tool.name: {weather_tool.name}")
print(f"weather_tool.description: {weather_tool.description[:60]}...")
print(f"calculator_tool.name: {calculator_tool.name}")
print(f"calculator_tool.description: {calculator_tool.description[:60]}...")


In [ ]:
# TODO 2: crear agente y executor

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the available tools to answer questions accurately."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

tools = [weather_tool, calculator_tool]

agent = create_tool_calling_agent(llm, tools, agent_prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5)

print(f"Agent: {type(agent).__name__}")
print(f"Executor: {type(executor).__name__}")
print("max_iterations=5 = el max_steps=5 de M3L1")
print("verbose=True = todos los prints de [Thought]/[Action]/[Obs] de M3L1")


In [ ]:
# TODO 3: invocar

print("=" * 55)
print("AGENTE LANGCHAIN (Estilo M3L2)")
print("=" * 55)
result = executor.invoke({"input": "What is the weather in Paris and what is 5 times that temperature?"})
print()
print(f"Respuesta final: {result['output']}")


In [ ]:
def run_checks():
    assert hasattr(weather_tool, 'name') and weather_tool.name == "weather_tool"
    assert hasattr(calculator_tool, 'name') and calculator_tool.name == "calculator_tool"
    assert weather_tool.description and len(weather_tool.description) > 20
    assert calculator_tool.description and len(calculator_tool.description) > 20
    assert agent is not None
    assert executor is not None
    result = executor.invoke({"input": "What is the weather in Paris?"})
    assert "output" in result and isinstance(result["output"], str)
    assert "21" in result["output"] or "celsius" in result["output"].lower()
    print("M3L2 E00 Resolution checks passed")

run_checks()


## Mapa final

| M3L1 (manual) | LangChain (M3L2) | Notebook M3L2 |
|---|---|---|
| f-string prompt | `ChatPromptTemplate` | E01 |
| `openai.chat...create()` | `ChatOpenAI` | E03 |
| `.choices[0].message.content` | `StrOutputParser` | E04 |
| Loop + trace manual | `AgentExecutor(verbose=True)` | Este |
| `def weather_tool()` | `@tool def weather_tool()` | Este |
| `max_steps=5` | `max_iterations=5` | Este |
| Retrieval naive | `FAISS` + `Retriever` | E06, E07 |
